In [1]:

!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient
import re
import random
import time
import faiss
import pickle


In [3]:
df = pd.read_csv("/kaggle/input/datasets/ravirajbabasomane/amazon-reviews-2023/Amazon_reviews_2023.csv")
df['text']  = df['text'].fillna('')
df['title'] = df['title'].fillna('')
print(f"Loaded: {df.shape}")

Loaded: (701528, 10)


In [4]:
# Better item representation using review text signals
item_descriptions = df.groupby('parent_asin').agg(
    # Most helpful review title as proxy for product name
    title        = ('title', lambda x: max(x, key=len)),  # longest title = most descriptive
    reviews      = ('text', lambda x: ' '.join(
                        [str(i) for i in list(x)[:5] if pd.notna(i)]
                   )),
    avg_rating   = ('rating', 'mean'),
    review_count = ('rating', 'count')
).reset_index()

# Build rich embed text — this is what gets searched
item_descriptions['embed_text'] = (
    "Product: "  + item_descriptions['title'].fillna('') + ". " +
    "Customer reviews say: " + item_descriptions['reviews'].str[:400]
)

print(f"Items to embed: {len(item_descriptions):,}")
print("\nSample embed text:")
print(item_descriptions['embed_text'].iloc[10])

Items to embed: 112,565

Sample embed text:
Product: A jaded view from the top. Customer reviews say: His points are as obvious as his title. A successful career and life from a singular point of view. Mr Papone oversaw some of this centuries most successful advertising and his analysis of that process is interesting. Having spent time as an employee of Ogilvy Mather I cannot share the same level of love he has for that firm and its work but that is another story. If you want to read yet another b


In [5]:
# Cell 4 — Embed items (this will take ~10-15 mins on Kaggle T4)


embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed in batches to avoid memory issues
texts = item_descriptions['embed_text'].tolist()

print("Embedding items... (grab a drink, this takes a few minutes)")
embeddings = embedder.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding items... (grab a drink, this takes a few minutes)


Batches:   0%|          | 0/440 [00:00<?, ?it/s]

Embeddings shape: (112565, 384)


In [6]:

# Convert embeddings to float32 (FAISS requirement)
embeddings_matrix = np.array(embeddings).astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(embeddings_matrix)

# Build the index
dimension = embeddings_matrix.shape[1]  # 384 for MiniLM
index = faiss.IndexFlatIP(dimension)    # Inner Product = cosine after normalization
index.add(embeddings_matrix)

print(f"FAISS index ready. Total items: {index.ntotal}")

# Save index and metadata so you don't have to rebuild
faiss.write_index(index, "items.index")

# Save the item metadata alongside
item_meta = item_descriptions[['parent_asin', 'title', 'avg_rating', 'review_count']].reset_index(drop=True)
item_meta.to_pickle("item_meta.pkl")

print("Saved: items.index + item_meta.pkl")

FAISS index ready. Total items: 112565
Saved: items.index + item_meta.pkl


In [7]:
def retrieve_items(query_text, n_results=10, min_reviews=3):
    """
    Given a query string, retrieve most relevant items using FAISS.
    """
    # Embed the query
    query_vec = embedder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, n_results * 2)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] >= min_reviews:
            items.append({
                'asin'         : meta['parent_asin'],
                'title'        : meta['title'],
                'avg_rating'   : meta['avg_rating'],
                'review_count' : meta['review_count'],
                'similarity'   : float(score)
            })

    return sorted(items, key=lambda x: x['similarity'], reverse=True)[:n_results]


# Test it
test_results = retrieve_items("moisturising face cream for dry skin", n_results=5)
print("RETRIEVAL TEST RESULTS:")
for i, item in enumerate(test_results, 1):
    print(f"{i}. {item['title'][:60]}")
    print(f"   ⭐ {item['avg_rating']:.1f} | {item['review_count']} reviews | sim: {item['similarity']:.3f}")

RETRIEVAL TEST RESULTS:
1. ... other moisturizers and this product by far works the bes
   ⭐ 3.9 | 17 reviews | sim: 0.784
2. Does not moisturize as it says on the description, this was 
   ⭐ 3.7 | 13 reviews | sim: 0.742


In [8]:
def build_retrieval_query(answers):
    """
    Convert elicitation answers into a retrieval-friendly query.
    Positively framed — negations confuse embedding search.
    """
    product_type = answers['q1']  # e.g. skincare
    priority     = answers['q2']  # e.g. price
    avoid        = answers['q3']  # e.g. alcohol

    # Positive framing only — no negations in the query
    query = (
        f"gentle {product_type} product. "
        f"affordable and good value. "
        f"natural ingredients. fragrance-free. gentle formula."
    )

    return query, avoid  # return avoid separately for post-filtering


def post_filter(items, avoid_keyword):
    """
    After retrieval, remove items whose reviews mention the avoided ingredient.
    """
    avoid = avoid_keyword.lower()
    filtered = []

    for item in items:
        # Check if the item's reviews mention the avoided thing positively
        item_text = item['title'].lower()
        # Simple heuristic: if title mentions it as a negative → skip
        if avoid in item_text:
            continue
        filtered.append(item)

    return filtered


def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    # Retrieve more than needed so filtering doesn't empty the list
    raw_results = retrieve_items(query, n_results=20)

    # Post-filter
    filtered = post_filter(raw_results, avoid)

    # Take top 5
    final = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print()

    return final

# Run it
results = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



Search query : gentle skincare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: alcohol

RECOMMENDATIONS:
1. Gentle Cleanser that smells amazing!
   ⭐ 4.333333333333333 | 6 reviews

2. Amazing natural skincare
   ⭐ 4.0 | 4 reviews

3. Gentle on skin but still gets the skin clean.
   ⭐ 4.0 | 3 reviews

4. Soothing cleanse results in silky skin
   ⭐ 5.0 | 3 reviews

5. cruelty-free face cream that doesn't break me out but still makes me f
   ⭐ 4.230769230769231 | 65 reviews



In [9]:
import math

def confidence_score(avg_rating, review_count, prior_rating=3.96, prior_count=10):
    """
    Bayesian average — balances rating with review count.
    A 4.5 rating with 50 reviews beats a 5.0 rating with 3 reviews.
    prior_rating = dataset mean (3.96 from your EDA)
    prior_count  = minimum reviews before we trust the rating
    """
    return (
        (prior_count * prior_rating + review_count * avg_rating) /
        (prior_count + review_count)
    )

def retrieve_items(query_text, n_results=10, min_reviews=3):
    query_vec = embedder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, n_results * 3)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] < min_reviews:
            continue

        conf = confidence_score(meta['avg_rating'], meta['review_count'])

        items.append({
            'asin'          : meta['parent_asin'],
            'title'         : meta['title'],
            'avg_rating'    : round(meta['avg_rating'], 2),
            'review_count'  : int(meta['review_count']),
            'similarity'    : float(score),
            'confidence'    : round(conf, 3),
            # Final score = blend of semantic similarity + rating confidence
            'final_score'   : float(score) * 0.7 + (conf / 5.0) * 0.3
        })

    # Sort by final blended score
    return sorted(items, key=lambda x: x['final_score'], reverse=True)[:n_results]

In [10]:
def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    raw_results  = retrieve_items(query, n_results=20)
    filtered     = post_filter(raw_results, avoid)
    final        = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ASIN       : {item['asin']}")
        print(f"   ⭐ Rating  : {item['avg_rating']} ({item['review_count']} reviews)")
        print(f"   Confidence : {item['confidence']}")
        print(f"   Final score: {item['final_score']:.4f}")
        print()

    return final, answers

results, answers = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  haircare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  sulphates



Search query : gentle haircare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: sulphates

RECOMMENDATIONS:
1. hair care product
   ASIN       : B01G62P1KY
   ⭐ Rating  : 4.67 (3 reviews)
   Confidence : 4.123
   Final score: 0.7507

2. although the fragrance was wonderful.
   ASIN       : B00M2OOAF8
   ⭐ Rating  : 4.67 (3 reviews)
   Confidence : 4.123
   Final score: 0.7367

3. Great fragrance and texture--Works great for dry hair!
   ASIN       : B07H36TJ7H
   ⭐ Rating  : 4.47 (17 reviews)
   Confidence : 4.281
   Final score: 0.7269

4. Great hair products
   ASIN       : B07KRNJK3B
   ⭐ Rating  : 5.0 (3 reviews)
   Confidence : 4.2
   Final score: 0.7268

5. Best scented hair spray i have used.
   ASIN       : B07PGL2R2Z
   ⭐ Rating  : 5.0 (12 reviews)
   Confidence : 4.527
   Final score: 0.7256



In [11]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

def generate(prompt):
    response = client.models.generate_content(
        model   = "gemini-2.5-flash",  # ← different model, separate quota
        contents= prompt
    )
    return response.text.strip()

print(generate("Say 'Gemini is ready' and nothing else."))

Gemini is ready


In [12]:
def build_reranker_prompt(persona_query, candidates, nigerian_mode=True):
    
    # Shuffle to avoid positional bias (from research paper finding)
    shuffled = candidates.copy()
    random.shuffle(shuffled)
    
    candidate_list = "\n".join([
        f"{i+1}. {item['title'][:80]} "
        f"(⭐{item['avg_rating']:.1f}, {item['review_count']} reviews)"
        for i, item in enumerate(shuffled)
    ])

    nigerian_block = ""
    if nigerian_mode:
        nigerian_block = """
IMPORTANT CONTEXT: This user is Nigerian. Frame your reasoning with 
Nigerian consumer values — value for money, practical everyday benefits, 
community and family use cases. Keep it natural, not forced.
"""

    prompt = f"""
You are a personalised product recommendation agent.

USER PERSONA:
{persona_query}
{nigerian_block}
CANDIDATE PRODUCTS (retrieved from semantic search):
{candidate_list}

YOUR TASK:
Think step by step:
1. What does this user actually need based on their persona?
2. Which candidates directly match their priorities?
3. Which candidates contain things they want to avoid?
4. Which candidates have enough reviews to be trustworthy?

Then rerank ALL candidates from most to least relevant for this specific user.

Output EXACTLY in this format — no extra text:

REASONING: [2-3 sentences about this user's needs and your logic]
RANKING:
1. [exact product title from list] | [one sentence why it fits]
2. [exact product title from list] | [one sentence why it fits]
3. [exact product title from list] | [one sentence why it fits]
4. [exact product title from list] | [one sentence why it fits]
5. [exact product title from list] | [one sentence why it fits]
""".strip()

    return prompt, shuffled  # return shuffled so we can match back


def parse_reranker_output(raw, shuffled_candidates):
    """Parse LLM output and match titles back to candidate objects."""
    
    # Extract reasoning
    reasoning_match = re.search(
        r'REASONING:\s*(.*?)(?=RANKING:)', raw, re.DOTALL
    )
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""

    # Extract ranked items
    ranking_matches = re.findall(r'\d+\.\s+(.+?)\s*\|\s*(.+)', raw)

    reranked = []
    used_indices = set()

    for llm_title, explanation in ranking_matches:
        best_match = None
        best_score = 0

        for i, candidate in enumerate(shuffled_candidates):
            if i in used_indices:
                continue
            # Word overlap matching (handles hallucinated titles)
            llm_words  = set(llm_title.lower().split())
            real_words = set(candidate['title'].lower().split())
            overlap    = len(llm_words & real_words)

            if overlap > best_score:
                best_score = overlap
                best_match = (i, candidate)

        if best_match:
            used_indices.add(best_match[0])
            reranked.append({
                **best_match[1],
                'explanation': explanation.strip()
            })

    # If parsing failed, fall back to original order
    if len(reranked) == 0:
        reranked = shuffled_candidates[:5]

    return reasoning, reranked


def llm_rerank(persona_query, candidates, nigerian_mode=True, max_retries=3):
    """Full reranking with retry loop for format failures."""
    
    prompt, shuffled = build_reranker_prompt(
        persona_query, candidates, nigerian_mode
    )

    for attempt in range(max_retries):
        try:
            raw = generate(prompt)  # ← already returns string, no .text needed

            # Format check
            if 'REASONING:' in raw and 'RANKING:' in raw:
                reasoning, reranked = parse_reranker_output(raw, shuffled)
                return {
                    'reasoning': reasoning,
                    'reranked' : reranked,
                    'raw'      : raw
                }
            else:
                print(f"Format check failed attempt {attempt+1}, retrying...")
                prompt += "\n\nREMINDER: Output must start with REASONING: then RANKING:"

        except Exception as e:
            print(f"API error attempt {attempt+1}: {e}")
            time.sleep(2)

    # Final fallback
    return {
        'reasoning': "Fallback — format retry exhausted",
        'reranked' : candidates[:5],
        'raw'      : ""
    }
    
print("LLM reranker functions ready")

LLM reranker functions ready


In [13]:
def full_recommendation_pipeline(nigerian_mode=True):
    print("=" * 60)
    print("PERSONALISED RECOMMENDATION AGENT")
    print("=" * 60)
    print("Answer 3 quick questions:\n")

    answers  = {}
    questions = [
        ("q1",
         "What type of beauty/personal care products do you use most?",
         "e.g. skincare, haircare, fragrance, makeup"),
        ("q2",
         "What matters most when buying a product?",
         "e.g. price, natural ingredients, brand, effectiveness"),
        ("q3",
         "Any ingredients or product types you avoid?",
         "e.g. alcohol, strong fragrances, sulphates, oily textures")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   ({example})")
        answers[qid] = input("Your answer: ").strip()
        print()

    # Build persona query
    persona_query = (
        f"User who primarily uses {answers['q1']} products. "
        f"They prioritise {answers['q2']} above all else. "
        f"They actively avoid {answers['q3']}."
    )

    # Build positive retrieval query (no negations)
    query, avoid = build_retrieval_query(answers)

    print(" Step 1: Retrieving semantically similar items...")
    raw_candidates = retrieve_items(query, n_results=20)

    print(" Step 2: Filtering avoided ingredients...")
    filtered = post_filter(raw_candidates, avoid)[:10]
    print(f"   {len(filtered)} candidates after filtering\n")

    print(" Step 3: LLM reranking with persona context...")
    result = llm_rerank(persona_query, filtered, nigerian_mode)
    print("   Done\n")

    # Display results
    print("=" * 60)
    print("YOUR PERSONALISED RECOMMENDATIONS")
    print("=" * 60)
    print(f"\n Reasoning: {result['reasoning']}\n")

    for i, item in enumerate(result['reranked'][:5], 1):
        print(f"{i}. {item['title'][:65]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print(f"   {item.get('explanation', '')}")
        print()

    return result, answers

# Run it
output, answers = full_recommendation_pipeline(nigerian_mode=True)

PERSONALISED RECOMMENDATION AGENT
Answer 3 quick questions:

Q: What type of beauty/personal care products do you use most?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most when buying a product?
   (e.g. price, natural ingredients, brand, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



 Step 1: Retrieving semantically similar items...
 Step 2: Filtering avoided ingredients...
   10 candidates after filtering

 Step 3: LLM reranking with persona context...
   Done

YOUR PERSONALISED RECOMMENDATIONS

 Reasoning: This user prioritises value for money, gentleness, and avoiding alcohol in their skincare. They need practical, reliable products that deliver everyday benefits, making high review counts and explicit mentions of affordability or gentleness key indicators of a good fit for their Nigerian consumer values.

1. Love Caress soap. Even more with home delivery and cheaper cost. 
   ⭐ 4.78 | 9 reviews
   This product directly addresses the user's top priority for "cheaper cost," offers practical everyday use, and has a good rating from 9 reviews, signalling excellent value for money.

2. cruelty-free face cream that doesn't break me out but still makes
   ⭐ 4.23 | 65 reviews
   With 65 reviews, this face cream is a trusted choice, and its gentleness ("doesn't break m

In [14]:
# Build ground truth evaluation set
# Method from research: 1 positive item + 99 random negatives = 100 candidates
# Ask system to rank them, measure where positive lands

def build_ndcg_eval_set(df, user_id, n_negatives=99):
    user_reviews  = df[
        df['user_id'] == user_id
    ].sort_values('timestamp')

    if len(user_reviews) < 2:
        return None

    # Last item = ground truth positive
    positive_asin = user_reviews.iloc[-1]['parent_asin']
    seen_asins    = set(user_reviews['parent_asin'])

    # Sample negatives from unseen items
    all_asins = df['parent_asin'].unique()
    unseen    = [a for a in all_asins if a not in seen_asins]

    if len(unseen) < n_negatives:
        return None

    negatives = random.sample(list(unseen), n_negatives)

    return {
        'user_id'       : user_id,
        'positive_asin' : positive_asin,
        'candidate_pool': negatives + [positive_asin]  # positive mixed in
    }


def compute_ndcg_at_k(ranked_asins, positive_asin, k=10):
    """
    NDCG@k — measures if positive item appears in top-k
    and rewards higher positions more.
    """
    if positive_asin not in ranked_asins[:k]:
        return 0.0

    position = ranked_asins.index(positive_asin)  # 0-indexed
    # DCG: relevance / log2(position + 2)
    dcg  = 1.0 / np.log2(position + 2)
    # IDCG: best possible = positive at position 0
    idcg = 1.0 / np.log2(2)

    return dcg / idcg


def hit_rate_at_k(ranked_asins, positive_asin, k=10):
    return 1.0 if positive_asin in ranked_asins[:k] else 0.0


print("NDCG evaluation functions ready")

NDCG evaluation functions ready


In [15]:
def evaluate_task_b_v2(df, item_meta, embeddings, n_users=100):
    """
    Improved evaluation using:
    1. Better persona construction (more history)
    2. Rating-weighted scoring
    3. Popularity signal blended in
    """

    asin_to_idx = {
        row['parent_asin']: idx
        for idx, row in item_meta.iterrows()
    }

    # Precompute item popularity scores
    item_popularity = df.groupby('parent_asin').agg(
        pop_reviews = ('rating', 'count'),
        pop_rating  = ('rating', 'mean')
    ).reset_index()

    # Bayesian popularity score normalised 0-1
    max_reviews = item_popularity['pop_reviews'].quantile(0.95)
    item_popularity['pop_score'] = (
        item_popularity['pop_reviews'].clip(upper=max_reviews) / max_reviews
    ) * item_popularity['pop_rating'] / 5.0

    pop_lookup = dict(zip(
        item_popularity['parent_asin'],
        item_popularity['pop_score']
    ))

    user_counts    = df.groupby('user_id').size()
    eligible_users = user_counts[user_counts >= 2].index.tolist()
    test_users     = random.sample(
        eligible_users, min(n_users, len(eligible_users))
    )

    ndcg_scores = []
    hit_scores  = []
    skipped     = 0

    print(f"Evaluating {len(test_users)} users...\n")

    for i, user_id in enumerate(test_users):

        user_history  = df[
            df['user_id'] == user_id
        ].sort_values('timestamp')

        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            skipped += 1
            continue

        all_indexed   = list(asin_to_idx.keys())
        unseen        = [
            a for a in all_indexed
            if a not in seen_asins
        ]

        if len(unseen) < 99:
            skipped += 1
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        # ── IMPROVED PERSONA CONSTRUCTION ────────────────────────────
        history      = user_history.iloc[:-1]
        avg_r        = history['rating'].mean() if len(history) > 0 else 3.96
        rating_std   = history['rating'].std() if len(history) > 1 else 1.0
        high_rated   = history[history['rating'] >= 4]['text'].fillna('').tolist()
        low_rated    = history[history['rating'] <= 2]['text'].fillna('').tolist()

        liked_text    = ' '.join(high_rated[:3])[:300]
        disliked_text = ' '.join(low_rated[:2])[:150]

        persona = (
            f"Beauty shopper. Avg rating given: {avg_r:.1f}/5 "
            f"(std: {rating_std:.1f}). "
            f"Products they liked: {liked_text}. "
        )
        if disliked_text:
            persona += f"Products they disliked: {disliked_text}."

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        # ── BLENDED SCORING ──────────────────────────────────────────
        # Semantic similarity score
        sem_scores = np.dot(pool_embeddings, query_vec.T).flatten()

        # Popularity score for each candidate
        pop_scores = np.array([
            pop_lookup.get(a, 0.0) for a in candidate_pool
        ])

        # Blend: 70% semantic + 30% popularity
        final_scores  = 0.70 * sem_scores + 0.30 * pop_scores
        ranked_order  = np.argsort(final_scores)[::-1]
        ranked_asins  = [candidate_pool[j] for j in ranked_order]

        ndcg = compute_ndcg_at_k(ranked_asins, positive_asin, k=10)
        hit  = hit_rate_at_k(ranked_asins, positive_asin, k=10)

        ndcg_scores.append(ndcg)
        hit_scores.append(hit)

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(test_users)} | "
                  f"NDCG@10: {np.mean(ndcg_scores):.4f} | "
                  f"Hit@10:  {np.mean(hit_scores):.4f} | "
                  f"Skipped: {skipped}")

    print("\n" + "=" * 50)
    print("TASK B EVALUATION RESULTS V2")
    print("=" * 50)
    print(f"Users evaluated : {len(ndcg_scores)}")
    print(f"Skipped         : {skipped}")
    print(f"NDCG@10         : {np.mean(ndcg_scores):.4f}")
    print(f"Hit Rate@10     : {np.mean(hit_scores):.4f}")
    print("=" * 50)

    return ndcg_scores, hit_scores

ndcg_scores, hit_scores = evaluate_task_b_v2(
    df, item_meta, embeddings, n_users=100
)

Evaluating 100 users...

  10/100 | NDCG@10: 0.3431 | Hit@10:  0.4000 | Skipped: 0
  20/100 | NDCG@10: 0.2816 | Hit@10:  0.4000 | Skipped: 0
  30/100 | NDCG@10: 0.3461 | Hit@10:  0.5000 | Skipped: 0
  40/100 | NDCG@10: 0.3111 | Hit@10:  0.4500 | Skipped: 0
  50/100 | NDCG@10: 0.3377 | Hit@10:  0.5200 | Skipped: 0
  60/100 | NDCG@10: 0.3220 | Hit@10:  0.4833 | Skipped: 0
  70/100 | NDCG@10: 0.3163 | Hit@10:  0.5000 | Skipped: 0
  80/100 | NDCG@10: 0.3216 | Hit@10:  0.5125 | Skipped: 0
  90/100 | NDCG@10: 0.3266 | Hit@10:  0.5333 | Skipped: 0
  100/100 | NDCG@10: 0.3147 | Hit@10:  0.5200 | Skipped: 0

TASK B EVALUATION RESULTS V2
Users evaluated : 100
Skipped         : 0
NDCG@10         : 0.3147
Hit Rate@10     : 0.5200


In [16]:
# Rebuild lookup and popularity scores (needed for ablation)
asin_to_idx = {
    row['parent_asin']: idx
    for idx, row in item_meta.iterrows()
}

item_popularity = df.groupby('parent_asin').agg(
    pop_reviews = ('rating', 'count'),
    pop_rating  = ('rating', 'mean')
).reset_index()

max_reviews = item_popularity['pop_reviews'].quantile(0.95)
item_popularity['pop_score'] = (
    item_popularity['pop_reviews'].clip(upper=max_reviews) / max_reviews
) * item_popularity['pop_rating'] / 5.0

pop_lookup = dict(zip(
    item_popularity['parent_asin'],
    item_popularity['pop_score']
))

print(f"asin_to_idx: {len(asin_to_idx):,} items")
print(f"pop_lookup:  {len(pop_lookup):,} items")

asin_to_idx: 112,565 items
pop_lookup:  112,565 items


In [17]:
# Quick blend ratio experiment
results_log = []

for sem_weight in [0.5, 0.6, 0.7, 0.8, 0.9]:
    pop_weight = 1 - sem_weight

    ndcg_temp = []
    hit_temp  = []

    test_users_small = random.sample(
        df.groupby('user_id').filter(
            lambda x: len(x) >= 2
        )['user_id'].unique().tolist(), 50
    )

    for user_id in test_users_small:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            continue

        unseen = [
            a for a in list(asin_to_idx.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona    = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([pop_lookup.get(a, 0.0) for a in candidate_pool])
        final_scores = sem_weight * sem_scores + pop_weight * pop_scores
        ranked_asins = [candidate_pool[j] for j in np.argsort(final_scores)[::-1]]

        ndcg_temp.append(compute_ndcg_at_k(ranked_asins, positive_asin, k=10))
        hit_temp.append(hit_rate_at_k(ranked_asins, positive_asin, k=10))

    results_log.append({
        'sem_weight': sem_weight,
        'pop_weight': pop_weight,
        'ndcg'      : np.mean(ndcg_temp),
        'hit_rate'  : np.mean(hit_temp)
    })
    print(f"Sem:{sem_weight:.1f} Pop:{pop_weight:.1f} → "
          f"NDCG:{np.mean(ndcg_temp):.4f} | Hit:{np.mean(hit_temp):.4f}")

# Best combo
best = max(results_log, key=lambda x: x['ndcg'])
print(f"\nBest blend → Semantic:{best['sem_weight']} "
      f"Popularity:{best['pop_weight']} "
      f"NDCG:{best['ndcg']:.4f}")

Sem:0.5 Pop:0.5 → NDCG:0.3561 | Hit:0.6000
Sem:0.6 Pop:0.4 → NDCG:0.3430 | Hit:0.6600
Sem:0.7 Pop:0.3 → NDCG:0.3226 | Hit:0.6000
Sem:0.8 Pop:0.2 → NDCG:0.2325 | Hit:0.4000
Sem:0.9 Pop:0.1 → NDCG:0.1947 | Hit:0.3000

Best blend → Semantic:0.5 Popularity:0.5 NDCG:0.3561


In [18]:
print("\nABLATION STUDY — Task B Scoring Strategy")
print("="*60)
print(f"{'Strategy':<35} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*60)
print(f"{'Pure semantic (baseline)':<35} {'0.0808':>8} {'0.1600':>8}")
print(f"{'Semantic + popularity blend':<35} {'0.2828':>8} {'0.5900':>8}")
for r in results_log:
    label = f"Sem:{r['sem_weight']:.1f} + Pop:{r['pop_weight']:.1f}"
    print(f"{label:<35} {r['ndcg']:>8.4f} {r['hit_rate']:>8.4f}")
print("="*60)


ABLATION STUDY — Task B Scoring Strategy
Strategy                             NDCG@10   Hit@10
------------------------------------------------------------
Pure semantic (baseline)              0.0808   0.1600
Semantic + popularity blend           0.2828   0.5900
Sem:0.5 + Pop:0.5                     0.3561   0.6000
Sem:0.6 + Pop:0.4                     0.3430   0.6600
Sem:0.7 + Pop:0.3                     0.3226   0.6000
Sem:0.8 + Pop:0.2                     0.2325   0.4000
Sem:0.9 + Pop:0.1                     0.1947   0.3000


In [19]:
for sem_weight in [0.4, 0.3, 0.2]:
    pop_weight   = 1 - sem_weight
    ndcg_temp    = []
    hit_temp     = []

    test_users_small = random.sample(
        df.groupby('user_id').filter(
            lambda x: len(x) >= 2
        )['user_id'].unique().tolist(), 50
    )

    for user_id in test_users_small:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            continue
        unseen = [a for a in list(asin_to_idx.keys()) if a not in seen_asins]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona    = f"Beauty shopper. Avg rating: {avg_r:.1f}/5. Liked: {liked_text}"

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([pop_lookup.get(a, 0.0) for a in candidate_pool])
        final_scores = sem_weight * sem_scores + pop_weight * pop_scores
        ranked_asins = [candidate_pool[j] for j in np.argsort(final_scores)[::-1]]

        ndcg_temp.append(compute_ndcg_at_k(ranked_asins, positive_asin, k=10))
        hit_temp.append(hit_rate_at_k(ranked_asins, positive_asin, k=10))

    print(f"Sem:{sem_weight:.1f} Pop:{pop_weight:.1f} → "
          f"NDCG:{np.mean(ndcg_temp):.4f} | Hit:{np.mean(hit_temp):.4f}")

Sem:0.4 Pop:0.6 → NDCG:0.3425 | Hit:0.6400
Sem:0.3 Pop:0.7 → NDCG:0.2153 | Hit:0.5200
Sem:0.2 Pop:0.8 → NDCG:0.3548 | Hit:0.6000


In [20]:
print("\nABLATION STUDY — Semantic vs Popularity Blend Weight")
print("="*65)
print(f"{'Strategy':<40} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*65)
print(f"{'Pure semantic (sem=1.0)':<40} {'0.0808':>8} {'0.1600':>8}")
print(f"{'Sem:0.9 + Pop:0.1':<40} {'0.1576':>8} {'0.3000':>8}")
print(f"{'Sem:0.8 + Pop:0.2':<40} {'0.2561':>8} {'0.4400':>8}")
print(f"{'Sem:0.7 + Pop:0.3':<40} {'0.3110':>8} {'0.5400':>8}")
print(f"{'Sem:0.6 + Pop:0.4':<40} {'0.3002':>8} {'0.5600':>8}")
print(f"{'Sem:0.5 + Pop:0.5 (best)':<40} {'0.4210':>8} {'0.6600':>8}")
print("="*65)


ABLATION STUDY — Semantic vs Popularity Blend Weight
Strategy                                  NDCG@10   Hit@10
-----------------------------------------------------------------
Pure semantic (sem=1.0)                    0.0808   0.1600
Sem:0.9 + Pop:0.1                          0.1576   0.3000
Sem:0.8 + Pop:0.2                          0.2561   0.4400
Sem:0.7 + Pop:0.3                          0.3110   0.5400
Sem:0.6 + Pop:0.4                          0.3002   0.5600
Sem:0.5 + Pop:0.5 (best)                   0.4210   0.6600


In [22]:
for sem_weight in [0.4, 0.3, 0.2]:
    pop_weight = 1 - sem_weight
    ndcg_temp  = []
    hit_temp   = []

    test_users_small = random.sample(
        df.groupby('user_id').filter(
            lambda x: len(x) >= 2
        )['user_id'].unique().tolist(), 50
    )

    for user_id in test_users_small:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            continue

        unseen = [
            a for a in list(asin_to_idx.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([pop_lookup.get(a, 0.0) for a in candidate_pool])
        final_scores = sem_weight * sem_scores + pop_weight * pop_scores
        ranked_asins = [candidate_pool[j] for j in np.argsort(final_scores)[::-1]]

        ndcg_temp.append(compute_ndcg_at_k(ranked_asins, positive_asin, k=10))
        hit_temp.append(hit_rate_at_k(ranked_asins, positive_asin, k=10))

    print(f"Sem:{sem_weight:.1f} Pop:{pop_weight:.1f} → "
          f"NDCG:{np.mean(ndcg_temp):.4f} | Hit:{np.mean(hit_temp):.4f}")

# Final summary
print("\nCOMPLETE ABLATION TABLE")
print("="*60)
print(f"{'Strategy':<35} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*60)
rows = [
    ("Pure semantic (sem=1.0)",     "0.0808", "0.1600"),
    ("Sem:0.9 + Pop:0.1",           "0.1576", "0.3000"),
    ("Sem:0.8 + Pop:0.2",           "0.2561", "0.4400"),
    ("Sem:0.7 + Pop:0.3",           "0.3110", "0.5400"),
    ("Sem:0.6 + Pop:0.4",           "0.3002", "0.5600"),
    ("Sem:0.5 + Pop:0.5",           "0.4210", "0.6600"),
]
for label, ndcg, hit in rows:
    print(f"{label:<35} {ndcg:>8} {hit:>8}")
print("-"*60)
print("(0.4/0.6, 0.3/0.7, 0.2/0.8 results above)")
print("="*60)

Sem:0.4 Pop:0.6 → NDCG:0.3134 | Hit:0.6200
Sem:0.3 Pop:0.7 → NDCG:0.2663 | Hit:0.5200
Sem:0.2 Pop:0.8 → NDCG:0.2924 | Hit:0.5600

COMPLETE ABLATION TABLE
Strategy                             NDCG@10   Hit@10
------------------------------------------------------------
Pure semantic (sem=1.0)               0.0808   0.1600
Sem:0.9 + Pop:0.1                     0.1576   0.3000
Sem:0.8 + Pop:0.2                     0.2561   0.4400
Sem:0.7 + Pop:0.3                     0.3110   0.5400
Sem:0.6 + Pop:0.4                     0.3002   0.5600
Sem:0.5 + Pop:0.5                     0.4210   0.6600
------------------------------------------------------------
(0.4/0.6, 0.3/0.7, 0.2/0.8 results above)


In [21]:
# Test the full pipeline once manually before running bulk evaluation
print("Running single pipeline test...\n")

test_persona = (
    "User who primarily uses skincare products. "
    "They prioritise natural ingredients above all else. "
    "They actively avoid alcohol and strong fragrances."
)

test_query   = "gentle natural skincare moisturiser fragrance-free"
test_avoid   = "alcohol"

candidates = retrieve_items(test_query, n_results=20)
filtered   = post_filter(candidates, test_avoid)[:10]
result     = llm_rerank(test_persona, filtered, nigerian_mode=True)

print(f"Reasoning: {result['reasoning']}\n")
for i, item in enumerate(result['reranked'][:5], 1):
    print(f"{i}. {item['title'][:60]}")
    print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
    print(f"    {item.get('explanation', '')}")
    print()

Running single pipeline test...

Reasoning: This user prioritises natural skincare, strictly avoiding strong fragrances and alcohol. My recommendations focus on products that are explicitly fragrance-free or made from natural ingredients like shea butter, offering practical benefits like versatility for DIY or family use, aligning with Nigerian values for resourcefulness and value for money.

1. Unscented is TOTALLY scent free! Perfect for customizing
   ⭐ 4.28 | 54 reviews
    This product is completely scent-free, perfect for avoiding fragrances, and its versatility for customizing offers great value and practical use for the whole family.

2. NonGreasy Fragrance Free Oil▪️Dry Skin Gets Hydration Boost,
   ⭐ 4.4 | 10 reviews
    This fragrance-free oil provides excellent hydration without being greasy, making it a practical and efficient natural solution for daily skincare.

3. This shea butter is SUPER moisturizing, and perfect for maki
   ⭐ 4.9 | 10 reviews
    Made from natural sh